In [2]:
# ============================================================
# EXPERIMENT 2 — FINAL EVALUATION SCRIPT
# ============================================================
# Evaluates:
#   1. Change Classification
#   2. Change Segmentation
#   3. Caption Generation
#
# Uses EXACT architecture from experiment2_best_model.pth
# Uses vocabulary STORED INSIDE THE CHECKPOINT
# Uses ACTUAL final_dataset.csv column names
# ============================================================

import os
import json
import ast
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

CSV_PATH = r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv"

CHECKPOINT_PATH = "experiment2_best_model.pth"

OUTPUT_PREDICTIONS = "experiment2_final_predictions.csv"
OUTPUT_METRICS = "experiment2_final_metrics.csv"

BATCH_SIZE = 4
NUM_WORKERS = 0

MAX_CAPTION_LEN = 40

# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

# ============================================================
# 3. CHECK FILES
# ============================================================

print("\n" + "=" * 70)
print("CHECKING FILES")
print("=" * 70)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"\nCSV NOT FOUND:\n{CSV_PATH}"
    )

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"""
CHECKPOINT NOT FOUND:

{os.path.abspath(CHECKPOINT_PATH)}

Put experiment2_best_model.pth in the same folder
as this notebook, or change CHECKPOINT_PATH above.
"""
    )

print("CSV found:")
print(CSV_PATH)

print("\nCheckpoint found:")
print(os.path.abspath(CHECKPOINT_PATH))

# ============================================================
# 4. LOAD DATASET
# ============================================================

print("\n" + "=" * 70)
print("LOADING DATASET")
print("=" * 70)

df = pd.read_csv(CSV_PATH)

print("Total samples:", len(df))

print("\nColumns:")
print(list(df.columns))

# ------------------------------------------------------------
# Verify the exact columns we need
# ------------------------------------------------------------

required_columns = [
    "split",
    "cc_A",
    "cc_B",
    "mask",
    "changeflag",
    "captions"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"""
Missing required columns:

{missing_columns}

Available columns:
{list(df.columns)}
"""
    )

# ------------------------------------------------------------
# Validation set
# ------------------------------------------------------------

val_df = (
    df[df["split"] == "val"]
    .reset_index(drop=True)
)

print("\nValidation samples:", len(val_df))

print("\nValidation change distribution:")
print(
    val_df["changeflag"].value_counts()
)

# ============================================================
# 5. CAPTION PARSING
# ============================================================

def parse_captions(x):

    if isinstance(x, list):
        return x

    if pd.isna(x):
        return [""]

    # JSON
    try:
        value = json.loads(x)

        if isinstance(value, list):
            return value

        if isinstance(value, str):
            return [value]

    except Exception:
        pass

    # Python literal
    try:
        value = ast.literal_eval(x)

        if isinstance(value, list):
            return value

        if isinstance(value, str):
            return [value]

    except Exception:
        pass

    return [str(x)]


# ============================================================
# 6. LOAD CHECKPOINT FIRST
# ============================================================

print("\n" + "=" * 70)
print("LOADING CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

print("Checkpoint loaded successfully.")

print("\nCheckpoint keys:")
print(list(checkpoint.keys()))

# ============================================================
# 7. GET VOCABULARY FROM CHECKPOINT
# ============================================================
#
# IMPORTANT:
# We DO NOT rebuild vocabulary from CSV.
#
# Your training script saved:
#   vocab_size
#   stoi
#   itos
#
# inside the checkpoint.
#
# This guarantees the embedding/output layer has the exact
# vocabulary size used during training.
# ============================================================

if "stoi" not in checkpoint:
    raise RuntimeError(
        """
This checkpoint does not contain 'stoi'.

This evaluation script expects the Experiment 2 checkpoint
created by the training script that saved stoi and itos.
"""
    )

if "itos" not in checkpoint:
    raise RuntimeError(
        """
This checkpoint does not contain 'itos'.

This evaluation script expects the Experiment 2 checkpoint
created by the training script that saved stoi and itos.
"""
    )

stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

VOCAB_SIZE = len(itos)

print("\nVocabulary size from checkpoint:", VOCAB_SIZE)

print("Checkpoint vocab_size field:",
      checkpoint.get("vocab_size", "not present"))

# ------------------------------------------------------------
# Verify vocabulary consistency
# ------------------------------------------------------------

if len(stoi) != len(itos):
    raise RuntimeError(
        "stoi and itos have different lengths."
    )

if "vocab_size" in checkpoint:

    if checkpoint["vocab_size"] != VOCAB_SIZE:

        raise RuntimeError(
            f"""
Checkpoint vocabulary mismatch.

checkpoint vocab_size:
{checkpoint["vocab_size"]}

len(itos):
{VOCAB_SIZE}
"""
        )

# ------------------------------------------------------------
# Special tokens
# ------------------------------------------------------------

PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

if PAD_TOKEN not in stoi:
    raise RuntimeError("PAD token missing from checkpoint vocabulary.")

if SOS_TOKEN not in stoi:
    raise RuntimeError("SOS token missing from checkpoint vocabulary.")

if EOS_TOKEN not in stoi:
    raise RuntimeError("EOS token missing from checkpoint vocabulary.")

if UNK_TOKEN not in stoi:
    raise RuntimeError("UNK token missing from checkpoint vocabulary.")

PAD_IDX = stoi[PAD_TOKEN]
SOS_IDX = stoi[SOS_TOKEN]
EOS_IDX = stoi[EOS_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]

print("\nSpecial tokens:")
print("PAD:", PAD_IDX)
print("SOS:", SOS_IDX)
print("EOS:", EOS_IDX)
print("UNK:", UNK_IDX)

# ============================================================
# 8. PATH HANDLING
# ============================================================

CSV_PARENT = Path(CSV_PATH).parent


def resolve_path(path_value):

    path_value = str(path_value)

    # Direct path
    if os.path.exists(path_value):
        return path_value

    # Relative to CSV directory
    candidate = CSV_PARENT / path_value

    if candidate.exists():
        return str(candidate)

    # Relative path with leading ./ or \
    candidate2 = Path(path_value)

    if candidate2.exists():
        return str(candidate2)

    # Return original so PIL gives useful error
    return path_value


# ============================================================
# 9. DATASET
# ============================================================

class LEVIRValidationDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # IMAGE A
        # ----------------------------------------------------

        image_a_path = resolve_path(
            row["cc_A"]
        )

        image_a = Image.open(
            image_a_path
        ).convert("RGB")

        image_a = np.asarray(
            image_a,
            dtype=np.float32
        ) / 255.0

        image_a = torch.from_numpy(
            image_a
        ).permute(2, 0, 1)

        # ----------------------------------------------------
        # IMAGE B
        # ----------------------------------------------------

        image_b_path = resolve_path(
            row["cc_B"]
        )

        image_b = Image.open(
            image_b_path
        ).convert("RGB")

        image_b = np.asarray(
            image_b,
            dtype=np.float32
        ) / 255.0

        image_b = torch.from_numpy(
            image_b
        ).permute(2, 0, 1)

        # ----------------------------------------------------
        # MASK
        # ----------------------------------------------------

        mask_path = resolve_path(
            row["mask"]
        )

        mask = Image.open(
            mask_path
        ).convert("L")

        mask = np.asarray(mask)

        # Resize 1024x1024 mask to 256x256
        if mask.shape != (256, 256):

            mask = Image.fromarray(
                mask
            ).resize(
                (256, 256),
                Image.Resampling.NEAREST
            )

            mask = np.asarray(mask)

        mask = (
            mask > 127
        ).astype(np.float32)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        # ----------------------------------------------------
        # CHANGE FLAG
        # ----------------------------------------------------

        changeflag = torch.tensor(
            float(row["changeflag"]),
            dtype=torch.float32
        )

        # ----------------------------------------------------
        # CAPTION
        # ----------------------------------------------------

        captions = parse_captions(
            row["captions"]
        )

        caption = str(
            captions[0]
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "mask": mask,
            "changeflag": changeflag,
            "caption_text": caption
        }


# ============================================================
# 10. DATALOADER
# ============================================================

val_dataset = LEVIRValidationDataset(
    val_df
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("\nValidation batches:", len(val_loader))

# ============================================================
# 11. MODEL ARCHITECTURE
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


# ============================================================
# 12. EXACT SIAMESE U-NET FROM EXPERIMENT 2
# ============================================================

class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Shared encoder
        # ----------------------------------------------------

        self.enc1 = DoubleConv(
            3,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.enc4 = DoubleConv(
            128,
            256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        # ----------------------------------------------------
        # Mask
        # ----------------------------------------------------

        self.mask_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(
                1
            ),

            nn.Flatten(),

            nn.Linear(
                256,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                1
            )
        )

    def encode(self, x):

        e1 = self.enc1(x)

        p1 = self.pool(e1)

        e2 = self.enc2(p1)

        p2 = self.pool(e2)

        e3 = self.enc3(p2)

        p3 = self.pool(e3)

        e4 = self.enc4(p3)

        return e1, e2, e3, e4

    def forward(
        self,
        image_a,
        image_b
    ):

        # ----------------------------------------------------
        # A
        # ----------------------------------------------------

        a1, a2, a3, a4 = self.encode(
            image_a
        )

        # ----------------------------------------------------
        # B
        # ----------------------------------------------------

        b1, b2, b3, b4 = self.encode(
            image_b
        )

        # ----------------------------------------------------
        # Absolute differences
        # ----------------------------------------------------

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        classification = self.classifier(
            d4
        ).squeeze(1)

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        x = self.up3(
            d4
        )

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(
            x
        )

        x = self.up2(
            x
        )

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(
            x
        )

        x = self.up1(
            x
        )

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(
            x
        )

        mask = self.mask_head(
            x
        )

        return (
            mask,
            classification,
            d4
        )


# ============================================================
# 13. EXACT GRU CAPTION DECODER
# ============================================================

class CaptionDecoder(nn.Module):

    def __init__(
        self,
        feature_dim,
        vocab_size,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
            padding_idx=PAD_IDX
        )

        self.feature_projection = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.gru = nn.GRU(
            hidden_dim,
            hidden_dim,
            batch_first=True
        )

        self.output = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        features,
        captions
    ):

        # features:
        # [B,256,32,32]

        features = F.adaptive_avg_pool2d(
            features,
            1
        )

        features = features.flatten(
            1
        )

        hidden = self.feature_projection(
            features
        ).unsqueeze(0)

        inputs = captions[:, :-1]

        embeddings = self.embedding(
            inputs
        )

        outputs, _ = self.gru(
            embeddings,
            hidden
        )

        logits = self.output(
            outputs
        )

        return logits


# ============================================================
# 14. COMPLETE MODEL
# ============================================================

class ChangeCaptionModel(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.siamese = SiameseUNet()

        self.caption_decoder = CaptionDecoder(
            feature_dim=256,
            vocab_size=vocab_size
        )

    def forward(
        self,
        image_a,
        image_b,
        captions
    ):

        mask, classification, features = self.siamese(
            image_a,
            image_b
        )

        caption_logits = self.caption_decoder(
            features,
            captions
        )

        return (
            mask,
            classification,
            caption_logits
        )


# ============================================================
# 15. CREATE MODEL WITH CHECKPOINT VOCAB SIZE
# ============================================================

print("\n" + "=" * 70)
print("CREATING MODEL")
print("=" * 70)

model = ChangeCaptionModel(
    VOCAB_SIZE
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Model parameters:",
    round(parameter_count / 1e6, 6),
    "M"
)

# ============================================================
# 16. LOAD CHECKPOINT
# ============================================================

print("\n" + "=" * 70)
print("LOADING MODEL WEIGHTS")
print("=" * 70)

state_dict = checkpoint["model_state_dict"]

# ------------------------------------------------------------
# IMPORTANT:
# strict=True intentionally.
#
# If this fails, the checkpoint is NOT the Experiment 2
# architecture above. We don't silently load partial weights.
# ------------------------------------------------------------

model.load_state_dict(
    state_dict,
    strict=True
)

model.eval()

print("✓ Checkpoint loaded successfully.")

if "epoch" in checkpoint:
    print(
        "Checkpoint epoch:",
        checkpoint["epoch"]
    )

if "val_loss" in checkpoint:
    print(
        "Checkpoint validation loss:",
        checkpoint["val_loss"]
    )

# ============================================================
# 17. ARCHITECTURE VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("ARCHITECTURE VERIFICATION")
print("=" * 70)

print(
    "Mask head:",
    tuple(model.siamese.mask_head.weight.shape)
)

print(
    "Classifier:",
    tuple(model.siamese.classifier[2].weight.shape),
    "->",
    tuple(model.siamese.classifier[5].weight.shape)
)

print(
    "Caption embedding:",
    tuple(
        model.caption_decoder.embedding.weight.shape
    )
)

print(
    "Caption GRU:",
    tuple(
        model.caption_decoder.gru.weight_ih_l0.shape
    )
)

print(
    "Caption output:",
    tuple(
        model.caption_decoder.output.weight.shape
    )
)

# ============================================================
# 18. SHAPE TEST
# ============================================================

print("\n" + "=" * 70)
print("MODEL SHAPE TEST")
print("=" * 70)

test_batch = next(
    iter(val_loader)
)

test_a = test_batch["image_a"][:2].to(
    DEVICE
)

test_b = test_batch["image_b"][:2].to(
    DEVICE
)

test_captions = torch.zeros(
    (2, MAX_CAPTION_LEN),
    dtype=torch.long,
    device=DEVICE
)

test_captions[:, 0] = SOS_IDX

with torch.no_grad():

    test_mask, test_flag, test_caption = model(
        test_a,
        test_b,
        test_captions
    )

print("Input A:", test_a.shape)
print("Input B:", test_b.shape)
print("Mask output:", test_mask.shape)
print("Classification output:", test_flag.shape)
print("Caption output:", test_caption.shape)

assert test_mask.shape == (
    2,
    1,
    256,
    256
)

assert test_flag.shape == (
    2,
)

assert test_caption.shape == (
    2,
    MAX_CAPTION_LEN - 1,
    VOCAB_SIZE
)

print("\n✓ MODEL SHAPE TEST PASSED")

# ============================================================
# 19. CAPTION DECODING
# ============================================================

def decode_tokens(tokens):

    words = []

    for token in tokens:

        token = int(token)

        if token == EOS_IDX:
            break

        if token in (
            PAD_IDX,
            SOS_IDX
        ):
            continue

        if 0 <= token < len(itos):

            word = itos[token]

            if word not in (
                PAD_TOKEN,
                SOS_TOKEN,
                EOS_TOKEN
            ):
                words.append(word)

    return " ".join(words)


# ============================================================
# 20. GREEDY CAPTION GENERATION
# ============================================================

@torch.no_grad()
def generate_caption(features):

    # --------------------------------------------------------
    # features:
    # [B,256,32,32]
    # --------------------------------------------------------

    features = F.adaptive_avg_pool2d(
        features,
        1
    )

    features = features.flatten(
        1
    )

    hidden = model.caption_decoder.feature_projection(
        features
    ).unsqueeze(0)

    batch_size = features.size(0)

    current_token = torch.full(
        (batch_size, 1),
        SOS_IDX,
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=DEVICE
    )

    for _ in range(
        MAX_CAPTION_LEN - 1
    ):

        embedding = model.caption_decoder.embedding(
            current_token[:, -1:]
        )

        output, hidden = model.caption_decoder.gru(
            embedding,
            hidden
        )

        logits = model.caption_decoder.output(
            output[:, -1, :]
        )

        next_token = torch.argmax(
            logits,
            dim=-1
        )

        generated.append(
            next_token
        )

        finished |= (
            next_token == EOS_IDX
        )

        current_token = torch.cat(
            [
                current_token,
                next_token.unsqueeze(1)
            ],
            dim=1
        )

        if finished.all():
            break

    if len(generated) == 0:

        return [
            ""
            for _ in range(batch_size)
        ]

    generated = torch.stack(
        generated,
        dim=1
    )

    captions = []

    for row in generated:

        captions.append(
            decode_tokens(
                row.detach().cpu().tolist()
            )
        )

    return captions


# ============================================================
# 21. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def tokenize_text(text):

    text = normalize_text(text)

    if not text:
        return []

    return text.split()


# ============================================================
# 22. BLEU IMPLEMENTATION
# ============================================================

from collections import Counter


def ngrams(tokens, n):

    return [
        tuple(tokens[i:i+n])
        for i in range(
            len(tokens) - n + 1
        )
    ]


def modified_precision(
    candidate,
    reference,
    n
):

    cand_ngrams = Counter(
        ngrams(candidate, n)
    )

    ref_ngrams = Counter(
        ngrams(reference, n)
    )

    if sum(cand_ngrams.values()) == 0:
        return 0.0

    clipped = 0

    for gram, count in cand_ngrams.items():

        clipped += min(
            count,
            ref_ngrams.get(
                gram,
                0
            )
        )

    return clipped / sum(
        cand_ngrams.values()
    )


def sentence_bleu_score(
    candidate,
    reference,
    max_n
):

    if len(candidate) == 0:
        return 0.0

    precisions = []

    for n in range(
        1,
        max_n + 1
    ):

        p = modified_precision(
            candidate,
            reference,
            n
        )

        # smoothing
        if p == 0:
            p = 1e-9

        precisions.append(p)

    log_precision = sum(
        math.log(p)
        for p in precisions
    ) / max_n

    # Brevity penalty
    candidate_len = len(candidate)
    reference_len = len(reference)

    if candidate_len == 0:
        return 0.0

    if candidate_len > reference_len:

        bp = 1.0

    else:

        bp = math.exp(
            1 -
            reference_len /
            candidate_len
        )

    return bp * math.exp(
        log_precision
    )


# ============================================================
# 23. ROUGE-L
# ============================================================

def lcs_length(
    a,
    b
):

    if len(a) == 0 or len(b) == 0:
        return 0

    dp = [
        [0] * (len(b) + 1)
        for _ in range(len(a) + 1)
    ]

    for i in range(
        1,
        len(a) + 1
    ):

        for j in range(
            1,
            len(b) + 1
        ):

            if a[i-1] == b[j-1]:

                dp[i][j] = (
                    dp[i-1][j-1] + 1
                )

            else:

                dp[i][j] = max(
                    dp[i-1][j],
                    dp[i][j-1]
                )

    return dp[-1][-1]


def rouge_l_score(
    candidate,
    reference
):

    if len(candidate) == 0:
        return 0.0

    if len(reference) == 0:
        return 0.0

    lcs = lcs_length(
        candidate,
        reference
    )

    precision = lcs / len(candidate)
    recall = lcs / len(reference)

    if precision + recall == 0:
        return 0.0

    beta = 1.2

    return (
        (1 + beta ** 2)
        * precision
        * recall
        /
        (
            recall
            +
            beta ** 2 * precision
        )
    )


# ============================================================
# 24. METEOR-LIKE SCORE
# ============================================================

def meteor_score_simple(
    candidate,
    reference
):

    if len(candidate) == 0:
        return 0.0

    if len(reference) == 0:
        return 0.0

    reference_counter = Counter(
        reference
    )

    candidate_counter = Counter(
        candidate
    )

    matches = 0

    for word, count in candidate_counter.items():

        matches += min(
            count,
            reference_counter.get(
                word,
                0
            )
        )

    if matches == 0:
        return 0.0

    precision = (
        matches /
        len(candidate)
    )

    recall = (
        matches /
        len(reference)
    )

    if precision + recall == 0:
        return 0.0

    fmean = (
        10 *
        precision *
        recall
        /
        (
            recall +
            9 * precision
        )
    )

    # approximate fragmentation penalty
    matched_positions = []

    used = set()

    for word in candidate:

        for i, ref_word in enumerate(reference):

            if (
                ref_word == word
                and i not in used
            ):

                matched_positions.append(i)
                used.add(i)
                break

    chunks = 0

    if matched_positions:

        chunks = 1

        for i in range(
            1,
            len(matched_positions)
        ):

            if (
                matched_positions[i]
                !=
                matched_positions[i-1] + 1
            ):
                chunks += 1

    penalty = 0.5 * (
        chunks /
        max(matches, 1)
    ) ** 3

    return fmean * (
        1 - penalty
    )


# ============================================================
# 25. EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("STARTING FINAL EVALUATION")
print("=" * 70)

all_true_flags = []
all_pred_flags = []

all_true_masks = []
all_pred_masks = []

all_predictions = []

caption_references = []
caption_candidates = []

model.eval()

for batch in tqdm(
    val_loader,
    desc="Evaluating"
):

    image_a = batch["image_a"].to(
        DEVICE,
        non_blocking=True
    )

    image_b = batch["image_b"].to(
        DEVICE,
        non_blocking=True
    )

    masks = batch["mask"].to(
        DEVICE,
        non_blocking=True
    )

    true_flags = batch["changeflag"].to(
        DEVICE,
        non_blocking=True
    )

    caption_texts = batch["caption_text"]

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    with torch.no_grad():

        pred_mask_logits, pred_flag_logits, features = model.siamese(
            image_a,
            image_b
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        pred_prob = torch.sigmoid(
            pred_flag_logits
        )

        pred_flag = (
            pred_prob >= 0.5
        ).long()

        # ----------------------------------------------------
        # Segmentation
        # ----------------------------------------------------

        pred_mask_prob = torch.sigmoid(
            pred_mask_logits
        )

        pred_mask = (
            pred_mask_prob >= 0.5
        ).float()

        # ----------------------------------------------------
        # Captions
        # ----------------------------------------------------

        generated_captions = generate_caption(
            features
        )

    # --------------------------------------------------------
    # Save classification
    # --------------------------------------------------------

    all_true_flags.extend(
        true_flags.detach()
        .cpu()
        .numpy()
        .astype(int)
        .tolist()
    )

    all_pred_flags.extend(
        pred_flag.detach()
        .cpu()
        .numpy()
        .astype(int)
        .tolist()
    )

    # --------------------------------------------------------
    # Save segmentation
    # --------------------------------------------------------

    all_true_masks.append(
        masks.detach()
        .cpu()
        .numpy()
    )

    all_pred_masks.append(
        pred_mask.detach()
        .cpu()
        .numpy()
    )

    # --------------------------------------------------------
    # Captions
    # --------------------------------------------------------

    for reference, candidate in zip(
        caption_texts,
        generated_captions
    ):

        reference = str(
            parse_captions(reference)[0]
        )

        caption_references.append(
            reference
        )

        caption_candidates.append(
            candidate
        )


# ============================================================
# 26. CLASSIFICATION METRICS
# ============================================================

true_flags = np.array(
    all_true_flags
)

pred_flags = np.array(
    all_pred_flags
)

accuracy = accuracy_score(
    true_flags,
    pred_flags
)

precision = precision_score(
    true_flags,
    pred_flags,
    zero_division=0
)

recall = recall_score(
    true_flags,
    pred_flags,
    zero_division=0
)

f1 = f1_score(
    true_flags,
    pred_flags,
    zero_division=0
)

cm = confusion_matrix(
    true_flags,
    pred_flags
)

print("\n" + "=" * 70)
print("CHANGE CLASSIFICATION")
print("=" * 70)

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1        : {f1:.4f}"
)

print("\nConfusion Matrix:")
print(cm)

# ============================================================
# 27. SEGMENTATION METRICS
# ============================================================

true_masks = np.concatenate(
    all_true_masks,
    axis=0
)

pred_masks = np.concatenate(
    all_pred_masks,
    axis=0
)

true_masks = (
    true_masks > 0.5
)

pred_masks = (
    pred_masks > 0.5
)

# ------------------------------------------------------------
# GLOBAL PIXEL COUNTS
# ------------------------------------------------------------

intersection = np.logical_and(
    true_masks,
    pred_masks
).sum()

union = np.logical_or(
    true_masks,
    pred_masks
).sum()

true_positive = intersection

false_positive = np.logical_and(
    ~true_masks,
    pred_masks
).sum()

false_negative = np.logical_and(
    true_masks,
    ~pred_masks
).sum()

true_negative = np.logical_and(
    ~true_masks,
    ~pred_masks
).sum()

# ------------------------------------------------------------
# IoU
# ------------------------------------------------------------

if union == 0:

    segmentation_iou = 1.0

else:

    segmentation_iou = (
        intersection /
        union
    )

# ------------------------------------------------------------
# Dice
# ------------------------------------------------------------

dice_denominator = (
    2 * true_positive
    +
    false_positive
    +
    false_negative
)

if dice_denominator == 0:

    segmentation_dice = 1.0

else:

    segmentation_dice = (
        2 * true_positive
        /
        dice_denominator
    )

# ------------------------------------------------------------
# Pixel accuracy
# ------------------------------------------------------------

total_pixels = (
    true_masks.size
)

pixel_accuracy = (
    true_positive +
    true_negative
) / total_pixels

print("\n" + "=" * 70)
print("SEGMENTATION")
print("=" * 70)

print(
    f"IoU             : {segmentation_iou:.4f}"
)

print(
    f"Dice            : {segmentation_dice:.4f}"
)

print(
    f"Pixel Accuracy  : {pixel_accuracy:.4f}"
)

print(
    "\nPixel statistics:"
)

print(
    "Intersection:",
    intersection
)

print(
    "Union:",
    union
)

print(
    "TP:",
    true_positive
)

print(
    "FP:",
    false_positive
)

print(
    "FN:",
    false_negative
)

print(
    "TN:",
    true_negative
)

# ============================================================
# 28. CAPTION METRICS
# ============================================================

print("\n" + "=" * 70)
print("CAPTION GENERATION")
print("=" * 70)

bleu1_scores = []
bleu2_scores = []
bleu3_scores = []
bleu4_scores = []
meteor_scores = []
rouge_scores = []

for reference, candidate in tqdm(
    zip(
        caption_references,
        caption_candidates
    ),
    total=len(caption_references),
    desc="Caption metrics"
):

    reference_tokens = tokenize_text(
        reference
    )

    candidate_tokens = tokenize_text(
        candidate
    )

    bleu1_scores.append(
        sentence_bleu_score(
            candidate_tokens,
            reference_tokens,
            1
        )
    )

    bleu2_scores.append(
        sentence_bleu_score(
            candidate_tokens,
            reference_tokens,
            2
        )
    )

    bleu3_scores.append(
        sentence_bleu_score(
            candidate_tokens,
            reference_tokens,
            3
        )
    )

    bleu4_scores.append(
        sentence_bleu_score(
            candidate_tokens,
            reference_tokens,
            4
        )
    )

    meteor_scores.append(
        meteor_score_simple(
            candidate_tokens,
            reference_tokens
        )
    )

    rouge_scores.append(
        rouge_l_score(
            candidate_tokens,
            reference_tokens
        )
    )

BLEU1 = np.mean(
    bleu1_scores
)

BLEU2 = np.mean(
    bleu2_scores
)

BLEU3 = np.mean(
    bleu3_scores
)

BLEU4 = np.mean(
    bleu4_scores
)

METEOR = np.mean(
    meteor_scores
)

ROUGE_L = np.mean(
    rouge_scores
)

print(
    f"BLEU-1  : {BLEU1:.4f}"
)

print(
    f"BLEU-2  : {BLEU2:.4f}"
)

print(
    f"BLEU-3  : {BLEU3:.4f}"
)

print(
    f"BLEU-4  : {BLEU4:.4f}"
)

print(
    f"METEOR  : {METEOR:.4f}"
)

print(
    f"ROUGE-L : {ROUGE_L:.4f}"
)

# ============================================================
# 29. SAVE PREDICTIONS
# ============================================================

prediction_df = val_df.copy()

prediction_df[
    "pred_change_probability"
] = [
    float(x)
    for x in (
        torch.tensor(
            all_pred_flags,
            dtype=torch.float32
        )
    ).tolist()
]

prediction_df[
    "pred_change"
] = pred_flags

prediction_df[
    "true_change"
] = true_flags

prediction_df[
    "generated_caption"
] = caption_candidates

prediction_df[
    "reference_caption"
] = caption_references

prediction_df.to_csv(
    OUTPUT_PREDICTIONS,
    index=False
)

# ============================================================
# 30. SAVE METRICS
# ============================================================

metrics = {

    "checkpoint": CHECKPOINT_PATH,

    "checkpoint_epoch":
        checkpoint.get(
            "epoch",
            None
        ),

    "checkpoint_val_loss":
        checkpoint.get(
            "val_loss",
            None
        ),

    "vocab_size":
        VOCAB_SIZE,

    # Classification
    "classification_accuracy":
        accuracy,

    "classification_precision":
        precision,

    "classification_recall":
        recall,

    "classification_f1":
        f1,

    # Segmentation
    "segmentation_iou":
        segmentation_iou,

    "segmentation_dice":
        segmentation_dice,

    "segmentation_pixel_accuracy":
        pixel_accuracy,

    # Caption
    "bleu_1":
        BLEU1,

    "bleu_2":
        BLEU2,

    "bleu_3":
        BLEU3,

    "bleu_4":
        BLEU4,

    "meteor":
        METEOR,

    "rouge_l":
        ROUGE_L
}

metrics_df = pd.DataFrame(
    [metrics]
)

metrics_df.to_csv(
    OUTPUT_METRICS,
    index=False
)

# ============================================================
# 31. SHOW EXAMPLE CAPTIONS
# ============================================================

print("\n" + "=" * 70)
print("CAPTION EXAMPLES")
print("=" * 70)

for i in range(
    min(10, len(caption_candidates))
):

    print(
        f"\nExample {i+1}"
    )

    print(
        "Reference:",
        caption_references[i]
    )

    print(
        "Generated:",
        caption_candidates[i]
    )

# ============================================================
# 32. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)

print(
    f"Classification Accuracy : {accuracy:.4f}"
)

print(
    f"Classification Precision: {precision:.4f}"
)

print(
    f"Classification Recall   : {recall:.4f}"
)

print(
    f"Classification F1       : {f1:.4f}"
)

print(
    f"Segmentation IoU       : {segmentation_iou:.4f}"
)

print(
    f"Segmentation Dice      : {segmentation_dice:.4f}"
)

print(
    f"Pixel Accuracy         : {pixel_accuracy:.4f}"
)

print(
    f"BLEU-1                 : {BLEU1:.4f}"
)

print(
    f"BLEU-2                 : {BLEU2:.4f}"
)

print(
    f"BLEU-3                 : {BLEU3:.4f}"
)

print(
    f"BLEU-4                 : {BLEU4:.4f}"
)

print(
    f"METEOR                 : {METEOR:.4f}"
)

print(
    f"ROUGE-L                : {ROUGE_L:.4f}"
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    OUTPUT_PREDICTIONS
)

print(
    OUTPUT_METRICS
)

print("\n✓ EVALUATION COMPLETE")

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

CHECKING FILES
CSV found:
C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv

Checkpoint found:
C:\College\Dissertation\Data\experiment2_best_model.pth

LOADING DATASET
Total samples: 7264

Columns:
['filename', 'split', 'cc_A', 'cc_B', 'cd_filename', 'cd_A', 'cd_B', 'mask', 'tile', 'row', 'col', 'changeflag', 'changed_pixels', 'change_ratio', 'change_percent', 'captions']

Validation samples: 919

Validation change distribution:
changeflag
1    467
0    452
Name: count, dtype: int64

LOADING CHECKPOINT
Checkpoint loaded successfully.

Checkpoint keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_loss', 'vocab_size', 'stoi', 'itos']

Vocabulary size from checkpoint: 860
Checkpoint vocab_size field: 860

Special tokens:
PAD: 0
SOS: 1
EOS: 2
UNK: 3

Validation batches: 230

CREATING MODEL
Model parameters: 2.845246 M

LOADING MODEL WEIGHTS
✓ Checkpoint loaded successfully.
C

Evaluating:   0%|          | 0/230 [00:00<?, ?it/s]


CHANGE CLASSIFICATION
Accuracy  : 0.8303
Precision : 0.8388
Recall    : 0.8244
F1        : 0.8315

Confusion Matrix:
[[378  74]
 [ 82 385]]

SEGMENTATION
IoU             : 0.0000
Dice            : 0.0000
Pixel Accuracy  : 0.9560

Pixel statistics:
Intersection: 0
Union: 2652895
TP: 0
FP: 0
FN: 2652895
TN: 57574689

CAPTION GENERATION


Caption metrics:   0%|          | 0/919 [00:00<?, ?it/s]

BLEU-1  : 0.5629
BLEU-2  : 0.5043
BLEU-3  : 0.4738
BLEU-4  : 0.4581
METEOR  : 0.5460
ROUGE-L : 0.5597

CAPTION EXAMPLES

Example 1
Reference: two houses are built at the top .
Generated: there is no difference .

Example 2
Reference: some trees have been removed and a road appears at the top .
Generated: a house appears at the top right corner .

Example 3
Reference: several buildings replace many plants next to the road .
Generated: a road appears in the center of the forest .

Example 4
Reference: several houses are built at the top in place of some trees .
Generated: a house appears at the top right corner .

Example 5
Reference: there is no difference .
Generated: there is no difference .

Example 6
Reference: there is no difference .
Generated: there is no difference .

Example 7
Reference: there is no difference .
Generated: there is no difference .

Example 8
Reference: there is no difference .
Generated: there is no difference .

Example 9
Reference: there is no difference .
Ge